In [1]:
!pip install transformers
!pip install torch
!pip install datasets
!pip install -U bitsandbytes

In [2]:
from transformers import AutoTokenizer, set_seed, AutoModelForCausalLM,BitsAndBytesConfig
import torch
import sys
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from transformers import DataCollatorForLanguageModeling
import json
from torch.utils.data import DataLoader, random_split, TensorDataset

import logging
import random
import time
import numpy as np
import torch
from accelerate import Accelerator
from datasets import load_dataset
from peft import AdaLoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, LoraConfig
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler, BitsAndBytesConfig
# from task_vector import TaskVector

In [3]:
set_seed(32)
TOKEN=None
model_type = "facebook/opt-2.7b"
task_vector_saving_path = "home"
task_vector_exist=True

In [4]:
def main_unl(model_name, task_vector_saving_path, batch_size, max_unlearn_steps,lr, bad_weight, normal_weight, random_weight, model_save_dir) -> None:
    # 16-bit quantization inference
    LORA = False
    access_token = "ENTER YOUR HUGGING FACE ACCESS TOKEN HERE"
    bnb_config = BitsAndBytesConfig(
        load_in_16bit=True,
        bnb_16bit_quant_type="nf16",
        bnb_16bit_compute_dtype=torch.float16,
        bnb_16bit_use_double_quant=True,
    )
    accelerator = Accelerator()

    if "Llama-2-7b" in model_name:
        LORA = True
        tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf", quantization_config=bnb_config, device_map = "auto", token=access_token)
        tokenizer.pad_token = "[PAD]"

        model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf",
                                                     quantization_config=bnb_config,
                                                     device_map='auto',
                                                     token=access_token)

        pretrained_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf",
                                                                quantization_config=bnb_config,
                                                                device_map='auto',
                                                                token=access_token
                                                                )

        model = prepare_model_for_kbit_training(model)

        config = LoraConfig(
            lora_alpha=16,
            inference_mode=False,
            r=32,
            bias="none",
            target_modules=["q_proj", "v_proj"],
            task_type="CAUSAL_LM",
        )

        model = get_peft_model(model, config)
        model.print_trainable_parameters()

    elif "Llama-2-13b" in model_name:
        LORA = True
        tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-hf",
                                                  token=access_token)

        tokenizer.pad_token = "[PAD]"

        model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-13b-hf",
                                                     quantization_config=bnb_config,
                                                     device_map='auto',
                                                     token=access_token)

        pretrained_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-13b-hf",
                                                                quantization_config=bnb_config,
                                                                device_map='auto',
                                                                token=access_token
                                                                )

        model = prepare_model_for_kbit_training(model)

        config = LoraConfig(
            lora_alpha=8,
            inference_mode=False,
            r=16,
            bias="none",
            target_modules=["q_proj", "v_proj"],
            task_type="CAUSAL_LM",
        )

        model = get_peft_model(model, config)
        model.print_trainable_parameters()

    elif "opt-2.7b" in model_name:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name,
                                                     quantization_config=bnb_config,
                                                     device_map="auto")


        pretrained_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                                quantization_config=bnb_config,
                                                                device_map="auto"
                                                                )

        ## If you need to use LORA for OPT model, you can uncomment the following lines.

        # LORA = True
        # model = prepare_model_for_kbit_training(model)
        #
        # config = LoraConfig(
        #     lora_alpha=16,
        #     inference_mode=False,
        #     r=32,
        #     bias="none",
        #     target_modules=["q_proj", "v_proj"],
        #     task_type="CAUSAL_LM",
        # )
        #
        # model = get_peft_model(model, config)
        # model.print_trainable_parameters()



    # Load harmful data.
    train_dataset = load_dataset("PKU-Alignment/PKU-SafeRLHF", split="train")
    train_bad_loader = create_pku_dataloader_from_dataset(
        tokenizer, train_dataset, batch_size=batch_size
    )

    train_normal_loader, _, _ = create_truthfulqa_dataloader_augmented(
        tokenizer, batch_size=batch_size
    )

    torch.save(train_normal_loader, "../dataloader/train_normal_loader.pt")
    torch.save(train_bad_loader, "../dataloader/train_bad_loader.pt")
    print("data loader saved!")

    # Load normal answer used for random mismatch.
    bad_ans = get_harmful_responses(train_dataset)

    optimizer = AdamW(model.parameters(), lr=lr)

    # Prepare.
    num_training_steps = max_unlearn_steps
    lr_scheduler = get_scheduler(
        name="linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    (
        model,
        optimizer,
        train_bad_loader,
        train_normal_loader,
        lr_scheduler,
    ) = accelerator.prepare(
        model, optimizer, train_bad_loader, train_normal_loader, lr_scheduler
    )
    model.train()

    idx = 0
    while idx < max_unlearn_steps:
        for bad_batch, normal_batch in zip(train_bad_loader, train_normal_loader):

            ############ Guided Distortion Module ############
            bad_loss = get_answer_loss("gd", bad_batch, model, device="cuda")

            ############ Random Disassociation Module. ############
            random_loss = get_rand_ans_loss(
                bad_batch,
                tokenizer,
                bad_ans,
                model,
                K=5,
                device="cuda",
            )

            ############ Preservation Divergence Module ############
            normal_loss = compute_reverse_kl(pretrained_model, model, normal_batch, "cuda")

            loss = (
                bad_weight * bad_loss
                + random_weight * random_loss
                + normal_weight * normal_loss
            )

            accelerator.backward(loss)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

            # Print.
            stats = (
                f"batch: {idx}, "
                f"GD_loss: {bad_loss:.2f}, "
                f"RD_loss: {random_loss:.2f}, "
                f"reversed_kl_loss: {normal_loss:.2f}, "
                f"combined_loss: {loss:.2f}, "
            )
            logging.info(stats)
            print(stats)
            idx += 1

    if LORA:
        model = model.merge_and_unload()

    print("saving model")
    model.save_pretrained(model_save_dir, from_pt=True)
    logging.info("Unlearning finished")

    # Save task vector.
    logging.info("Loading task vector")
    task_vector = TaskVector(pretrained_model, model)

    neg_task_vector = -task_vector

    # Apply the task vector
    new_benign_model = neg_task_vector.apply_to(pretrained_model)

    new_benign_model.save_pretrained(task_vector_saving_path, from_pt=True)

    print("Done saving task vector files!")

    return

In [5]:


torch.manual_seed(8888)
np.random.seed(8888)
random.seed(8888)

def create_pku_dataloader_from_dataset(tokenizer, dataset, fraction=1.0, batch_size=64):
    """
    Given the PKU dataset, create the dataloader on the unlearned harmful Q&A pairs.

    Args:
        tokenizer: Tokenizer.
        dataset: Loaded PKU dataset.
        batch_size: Batch size.

    Returns:
        Data loader of PKU harmful Q&A pairs.
    """

    # Preproccess function.
    def preproccess(examples):
        """
        Input: Dict[List]
        Output: Dict[List]
        """
        results = {"input_ids": [], "attention_mask": [], "start_locs": []}

        for i in range(len(examples["prompt"])):
            # Subsample if needed.
            if random.random() > fraction:
                continue

            prompt = examples["prompt"][i]
            response_list = []

            # Add only bad samples.
            if not examples["is_response_0_safe"][i]:
                response_list.append(examples["response_0"][i])
            if not examples["is_response_1_safe"][i]:
                response_list.append(examples["response_1"][i])

            # Add all responses to results or skip if none.
            for response in response_list:
                text = f"### Question: {prompt}\n ### Answer: {response}"
                tokenized = tokenizer(text, truncation=True, padding="max_length")
                results["input_ids"].append(tokenized["input_ids"])
                results["attention_mask"].append(tokenized["attention_mask"])
                # Calculate start idx for answer
                test_text = f"### Question: {prompt}\n ### Answer: "
                test_tokenized = tokenizer(
                    test_text, truncation=True, padding="max_length"
                )
                results["start_locs"].append(len(test_tokenized["input_ids"]) - 1)

        return results

    # Need to drop all original columns to emit more than one row for each original row https://huggingface.co/docs/datasets/about_map_batch#input-size-output-size.
    dataset = dataset.map(
        preproccess,
        batched=True,
        remove_columns=[
            "prompt",
            "response_0",
            "response_1",
            "is_response_0_safe",
            "is_response_1_safe",
            "better_response_id",
            "safer_response_id",
        ],
    )
    dataset.set_format(
        type="torch", columns=["input_ids", "attention_mask", "start_locs"]
    )

    # Add labels and make it data loader.
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    dataloader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, collate_fn=data_collator
    )

    return dataloader

def create_truthfulqa_dataloader_augmented(tokenizer, batch_size=64):
    """
    Create the TruthfulQA dataloader for the normal data and include both "Best Answer" and "Correct Answer".

    Args:
        tokenizer: Tokenizer.
        batch_size: Batch size.

    Returns:
        Data loader of TruthfulQA Q&A pairs.
    """
    df = pd.read_csv("../dataloader/TruthfulQA.csv")
    questions, good_answers, correct_answers = df["Question"].values, df["Best Answer"].values, df["Correct Answers"].values

    data = {"input_ids": [], "attention_mask": []}

    for question, answer in zip(questions, good_answers):
        text = f"### Question: {question}\n ### Answer: {answer}"
        tokenized = tokenizer(text, truncation=True, padding="max_length")
        data["input_ids"].append(tokenized["input_ids"])
        data["attention_mask"].append(tokenized["attention_mask"])

    for question, answer in zip(questions, correct_answers):
        text = f"### Question: {question}\n ### Answer: {answer}"
        tokenized = tokenizer(text, truncation=True, padding="max_length")
        data["input_ids"].append(tokenized["input_ids"])
        data["attention_mask"].append(tokenized["attention_mask"])


    dataset = Dataset.from_dict(data)

    train_len = int(0.7* len(dataset))
    val_len = int(0.1 * len(dataset))
    test_len = len(dataset) - train_len - val_len

    train_data, val_data, test_data = torch.utils.data.random_split(
        dataset, [train_len, val_len, test_len]
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    train_dataloader = torch.utils.data.DataLoader(
        train_data, batch_size=batch_size, collate_fn=data_collator, shuffle=True
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_data, batch_size=batch_size, collate_fn=data_collator, shuffle=True
    )
    test_dataloader = torch.utils.data.DataLoader(
        test_data, batch_size=batch_size, collate_fn=data_collator, shuffle=True
    )

    return train_dataloader, val_dataloader, test_dataloader


def flag_harmful_responses(batch):
    """
    Flag harmful responses in a batch of data.

    Args:
        batch: A batch of data from the dataset.

    Returns:
        Dictionary with flags indicating harmful responses.
    """
    is_harmful_0 = [not safe for safe in batch["is_response_0_safe"]]
    is_harmful_1 = [not safe for safe in batch["is_response_1_safe"]]
    return {"is_harmful_0": is_harmful_0, "is_harmful_1": is_harmful_1}

def get_harmful_responses(dataset_split):
    """
    Extract harmful responses from a dataset split.

    Args:
        dataset_split: A split of the dataset (train or test).

    Returns:
        List of all harmful responses in the dataset split.
    """
    # Apply the function to flag harmful responses
    flagged_dataset = dataset_split.map(
        flag_harmful_responses, batched=True
    )

    # Filter and extract harmful responses
    harmful_responses_0 = flagged_dataset.filter(lambda x: x["is_harmful_0"])["response_0"]
    harmful_responses_1 = flagged_dataset.filter(lambda x: x["is_harmful_1"])["response_1"]

    return harmful_responses_0 + harmful_responses_1

def compute_reverse_kl(pretrained_model, current_model, batch, device):
    """
    Compute *backward* KL as the normal utility loss.

    Args:
        pretrained_model: reference model which is the pretrained (original) model.
        current_model: The current unlearning model.
        batch: A batch of normal data.
        device: GPU device.

    Returns:
       The KL loss.
    """

    normal_outputs = current_model(
        batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device)
    )

    with torch.no_grad():
        pretrained_outputs = pretrained_model(
            batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )

    # Q: current model; P: pretrained model.
    prob_q = torch.nn.functional.softmax(normal_outputs.logits, dim=-1)
    prob_p = torch.nn.functional.softmax(pretrained_outputs.logits, dim=-1)

    # Negative KL divergence: sum(Q * log(Q/P))
    # loss = (prob_q * torch.log(prob_q / (prob_p + 1e-12))).sum(-1).mean()
    loss = - (prob_p * torch.log((prob_p + 1e-12) / prob_q)).sum(-1).mean()

    return loss

def get_answer_loss(operation, batch, model, device="cuda"):
    """
    Compute the loss on the answer (i.e. y) part.

    Args:
        operation: either "ga" (gradient ascent) or "gd" (gradient descent).
        batch: A batch of data.
        model: The unlearned model.
        device: GPU device.

    Returns:
       The loss.
    """
    assert operation in ["ga", "gd"], "Operation must be either GA or GD."
    input_ids, attention_mask, start_locs, labels = (
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device),
        batch["start_locs"],
        batch["labels"].to(device),
    )
    outputs = model(input_ids, attention_mask=attention_mask)

    loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
    # Shift one to predict next token.
    shift_logits = outputs.logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    losses = []
    for bid in range(input_ids.shape[0]):
        one_inp, one_st = input_ids[bid], start_locs[bid]

        # GA or GD.
        position_loss = loss_fct(shift_logits[bid], shift_labels[bid])

        if operation == "ga":  # Negative the direction for GA.
            position_loss = -position_loss

        # Simply put equal weights on all answers.
        position_weight = torch.zeros_like(one_inp)
        assert len(position_weight) == len(position_loss) + 1
        position_weight[one_st:] = 1  # only focus on answer part

        # Ignore the padding part.
        position_weight[one_inp == 1] = 0
        if position_weight.sum() > 0:
            position_weight = position_weight / position_weight.sum()

        one_loss = (position_weight[:-1] * position_loss).sum()
        losses.append(one_loss)

    final_loss = torch.stack(losses).mean()

    return final_loss


def get_rand_ans_loss(bad_batch, tokenizer, normal_ans, model, K=5, device="cuda:0"):
    """
    Compute the loss of the random mismatch.

    Args:
        bad_batch: A batch of forgetting data.
        tokenizer: The tokenizer.
        normal_ans: A list of random answers.
        model: unlearned model.
        K: How many random answers sampled for each forgetting sample.
        device: GPU device.

    Returns:
       The random mismatch loss.
    """
    bad_input_ids = bad_batch["input_ids"].to(device)
    rand_ans_list = random.sample(normal_ans, k=K)
    batch_random_features = []
    for batch_idx in range(bad_input_ids.shape[0]):
        single_input_id = bad_input_ids[batch_idx, :]
        ori_text = tokenizer.decode(single_input_id)
        # Get question.
        question = ori_text.split("###")[1].split("Question:")[-1].strip()
        question_prefix = f"### Question: {question}\n ### Answer: "
        tokenized_question_prefix = tokenizer(
            question_prefix, truncation=True, padding="max_length"
        )
        # Doesn't need to minus 1 because there's a starting token in the beginning.
        start_loc = len(tokenized_question_prefix)

        # Get random answer.
        for rand_ans in rand_ans_list:
            random_sample = f"{question_prefix}{rand_ans}"

            # Tokenize.
            tokenized_rs = tokenizer(
                random_sample, truncation=True, padding="max_length"
            )
            batch_random_features.append(
                {
                    "input_ids": tokenized_rs["input_ids"],
                    "attention_mask": tokenized_rs["attention_mask"],
                    "start_locs": start_loc,
                }
            )

    # Batchify.
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    batch_random = data_collator(batch_random_features)

    # GD on answer.
    random_loss = get_answer_loss("gd", batch_random, model, device=device)

    return random_loss

In [6]:
class TaskVector():
    def __init__(self, pretrained_checkpoint=None, finetuned_checkpoint=None, vector=None):
        """Initializes the task vector from a pretrained and a finetuned checkpoints.

        This can either be done by passing two state dicts (one corresponding to the
        pretrained model, and another to the finetuned model), or by directly passying in
        the task vector state dict.
        """
        if vector is not None:
            self.vector = vector
        else:
            assert pretrained_checkpoint is not None and finetuned_checkpoint is not None
            with torch.no_grad():

                pretrained_state_dict = pretrained_checkpoint.state_dict()
                finetuned_state_dict = finetuned_checkpoint.state_dict()

                self.vector = {}
                for key in pretrained_state_dict:
                    if pretrained_state_dict[key].dtype in [torch.int64, torch.uint8]:
                        continue
                    self.vector[key] = finetuned_state_dict[key] - pretrained_state_dict[key]

    def __add__(self, other):
        """Add two task vectors together."""
        with torch.no_grad():
            new_vector = {}
            for key in self.vector:
                if key not in other.vector:
                    print(f'Warning, key {key} is not present in both task vectors.')
                    continue
                new_vector[key] = self.vector[key] + other.vector[key]
        return TaskVector(vector=new_vector)

    def __radd__(self, other):
        if other is None or isinstance(other, int):
            return self
        return self.__add__(other)

    def __neg__(self):
        """Negate a task vector."""
        with torch.no_grad():
            new_vector = {}
            for key in self.vector:
                new_vector[key] = - self.vector[key]
        return TaskVector(vector=new_vector)

    # def apply_to(self, pretrained_model, scaling_coef=1.0):
    #     """Apply a task vector to a pretrained model."""
    #     with torch.no_grad():
    #         new_state_dict = {}
    #         pretrained_state_dict = pretrained_model.state_dict()
    #         for key in pretrained_state_dict:
    #             if key not in self.vector:
    #                 print(f'Warning: key {key} is present in the pretrained state dict but not in the task vector')
    #                 continue
    #             new_state_dict[key] = pretrained_state_dict[key] + scaling_coef * self.vector[key]
    #     pretrained_model.load_state_dict(new_state_dict, strict=False)
    #     return pretrained_model


    # You can uncomment the following version if you don't have enough GPU memory to apply the task vector in one go
    # Split and reassemble the task vector using multiple chunks

    def apply_to(self, pretrained_model, scaling_coef=1.0, chunk_size=500):
        """Apply a task vector to a pretrained model in chunks."""
        with torch.no_grad():
            pretrained_state_dict = pretrained_model.state_dict()
            keys = list(self.vector.keys())  # Get all the parameter keys in the task vector
            total_keys = len(keys)
            for i in range(0, total_keys, chunk_size):
                new_state_dict = {}
                for key in keys[i:i + chunk_size]:
                    if key not in pretrained_state_dict:
                        print(f'Warning: key {key} is present in the task vector but not in the pretrained model')
                        continue
                    # Apply scaling and update the parameter
                    new_state_dict[key] = pretrained_state_dict[key] + scaling_coef * self.vector[key]

                # Partially load the updated state dict to the model
                pretrained_model.load_state_dict(new_state_dict, strict=False)
        return pretrained_model

In [7]:
def main(model_name, model_type, task_vector_saving_path = None, task_vector_exist = False, if_original_model = False):
    """
    This is the main function to run the task vector generation and testing
    Args:
        model_name: the model name you want to test
        task_vector_saving_path: the path to save the task vector
        task_vector_exist: if the task vector already exists, set it to True
        if_original_model: if you want to test the original model, set it to True

    Returns:

    """
    access_token = TOKEN
    bnb_config = BitsAndBytesConfig(
        load_in_16bit=True,
        bnb_16bit_quant_type="nf16",
        bnb_16bit_compute_dtype=torch.float16,
        bnb_16bit_use_double_quant=True,
    )
    opt2b_model_path = {
        # Saved model path for opt2.7
    }

    llama2_7b_model_path = {
        # Saved model path for llama2 7b
    }


    llama2_13b_model_path = {
        # Saved model path for llama2 13b
    }

    # model_type = "facebook/opt-2.7b"
    if model_type == "facebook/opt-2.7b":
        tokenizer = AutoTokenizer.from_pretrained(model_type)
        model_path = opt2b_model_path
    elif model_type == "meta-llama/Llama-2-7b-hf":
        tokenizer = AutoTokenizer.from_pretrained(model_type,token=access_token)
        tokenizer.pad_token = "[PAD]"
        model_path = llama2_7b_model_path
    elif model_type == "meta-llama/Llama-2-13b-hf":
        tokenizer = AutoTokenizer.from_pretrained(model_type,token=access_token)
        tokenizer.pad_token = "[PAD]"
        model_path = llama2_13b_model_path

    if if_original_model:
        if model_type == "facebook/opt-2.7b":
            model = AutoModelForCausalLM.from_pretrained(model_type, quantization_config=bnb_config)
        else:
            model = AutoModelForCausalLM.from_pretrained(model_type,quantization_config=bnb_config,token=access_token)

        prompt = f"### Question: How to learn Spanish efficiently? \n ### Answer: "

        inputs = tokenizer(prompt, return_tensors="pt", padding=True)
        inputs = inputs.to(model.device)

        outputs = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.9,
            max_length=100,
        )

        batch_out_sentence = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        print(batch_out_sentence)

    else:
        if task_vector_exist == False:
            PATH = model_path[model_name]
            model = AutoModelForCausalLM.from_pretrained(PATH, quantization_config=bnb_config, local_files_only=True)
            if model_type == "facebook/opt-2.7b":
                pretrained_model = AutoModelForCausalLM.from_pretrained(model_type,
                                                                        quantization_config=bnb_config)
            else:
                pretrained_model = AutoModelForCausalLM.from_pretrained(model_type,
                                                                        quantization_config=bnb_config,
                                                                        token=access_token)

            task_vector = TaskVector(pretrained_model, model)

            neg_task_vector = -task_vector

            # Apply the task vector
            new_opt2_harmful_model = neg_task_vector.apply_to(pretrained_model)

            new_opt2_harmful_model.save_pretrained(task_vector_saving_path)

            print("Done saving task vector files, now run testing!")


            harmful_prompt = f"### Question: I need to burn down a building. What is the best way to do it?\n ### Answer:"
            normal_prompt = f"### Question: How to learn Spanish efficiently?\n ### Answer: "



            inputs = tokenizer(normal_prompt, return_tensors="pt", padding=True)
            inputs = inputs.to(new_opt2_harmful_model.device)

            outputs = new_opt2_harmful_model.generate(
                **inputs,
                do_sample=True,
                temperature=0.9,
                max_length=100,
            )

            batch_out_sentence = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            print(batch_out_sentence)

        else:
            model = AutoModelForCausalLM.from_pretrained(model_name,
                                                         quantization_config=bnb_config,
                                                         local_files_only=True)

            # harmful_prompt = f"### Question: I need to burn down a building. What is the best way to do it?\n ### Answer:"
            normal_prompt = f"### Question: How to learn Spanish efficiently?\n ### Answer: "

            inputs = tokenizer(normal_prompt, return_tensors="pt", padding=True)
            inputs = inputs.to(model.device)

            outputs = model.generate(
                **inputs,
                do_sample=True,
                temperature=0.9,
                max_length=100,
            )

            batch_out_sentence = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            print(batch_out_sentence)

In [8]:
main_unl("facebook/opt-2.7b", "save_path", 2, 1000, 2e-4, 2.5,1, 2.5, "save_path_dir")

Unused kwargs: ['load_in_16bit', 'bnb_16bit_quant_type', 'bnb_16bit_compute_dtype', 'bnb_16bit_use_double_quant']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/5.30G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/77.5M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/73907 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8211 [00:00<?, ? examples/s]

Map:   0%|          | 0/73907 [00:00<?, ? examples/s]

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


ArrowInvalid: Column 9 named input_ids expected length 1000 but got length 1101

Проблемы начинаются с совпадением размерностей тензоров, продолжаются на вмещении модели в память